# 04 — Engineer Time Series Features
Builds weekly features from scored articles for forecasting and anomaly detection.
Input: `data/scored_articles.csv` → Outputs: `data/timeseries.csv`, `data/timeseries_wide.csv`

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/scored_articles.csv', parse_dates=['date'])
df = df.sort_values('date')

# Backward-compatible defaults
if 'primary_topic' not in df.columns:
    df['primary_topic'] = 'other'
if 'risk_score' not in df.columns:
    df['risk_score'] = 0
if 'risk_index' not in df.columns:
    df['risk_index'] = 0.0

df['risk_score'] = pd.to_numeric(df['risk_score'], errors='coerce').fillna(0)
df['risk_index'] = pd.to_numeric(df['risk_index'], errors='coerce').fillna(0.0)
print(df.shape)
print(df.columns.tolist())

(103343, 18)
['id', 'date', 'section', 'headline', 'body', 'wordcount', 'url', 'subtopic', 'topics', 'primary_topic', 'countries', 'organizations', 'commodities', 'risk_score', 'sentiment', 'sentiment_label', 'sentiment_intensity', 'risk_index']


In [3]:
# Weekly aggregation per primary topic
# If subtopic exists, you can later do a subtopic-level variant similarly.
df['week'] = df['date'].dt.to_period('W').dt.start_time

ts = (
    df.groupby(['week', 'primary_topic'])
    .agg(
        article_count=('id', 'count'),
        sentiment_mean=('sentiment', 'mean'),
        sentiment_std=('sentiment', 'std'),
        positive_ratio=('sentiment_label', lambda x: (x == 'positive').mean()),
        negative_ratio=('sentiment_label', lambda x: (x == 'negative').mean()),
        risk_score_mean=('risk_score', 'mean'),
        risk_score_max=('risk_score', 'max'),
        risk_index_mean=('risk_index', 'mean'),
        high_risk_share=('risk_score', lambda x: (x >= 7).mean()),
    )
    .reset_index()
)

ts['sentiment_std'] = ts['sentiment_std'].fillna(0)

# Time-series engineered features (per topic): momentum + rolling baselines
# These are useful for both forecasting and anomaly scoring.
ts = ts.sort_values(['primary_topic', 'week']).reset_index(drop=True)
ts['article_count_lag1'] = ts.groupby('primary_topic')['article_count'].shift(1)
ts['article_count_lag4'] = ts.groupby('primary_topic')['article_count'].shift(4)
ts['article_count_pct_change_1w'] = (
    (ts['article_count'] - ts['article_count_lag1'])
    / ts['article_count_lag1'].replace(0, np.nan)
)
ts['article_count_roll4_mean'] = (
    ts.groupby('primary_topic')['article_count']
      .transform(lambda s: s.rolling(4, min_periods=1).mean())
)
ts['risk_index_roll4_mean'] = (
    ts.groupby('primary_topic')['risk_index_mean']
      .transform(lambda s: s.rolling(4, min_periods=1).mean())
)

print(ts.shape)
ts.head(10)

(1299, 16)


,week,primary_topic,article_count,sentiment_mean,sentiment_std,positive_ratio,negative_ratio,risk_score_mean,risk_score_max,risk_index_mean,high_risk_share,article_count_lag1,article_count_lag4,article_count_pct_change_1w,article_count_roll4_mean,risk_index_roll4_mean
0,2022-07-18,ai,3,-0.9246,0.000000,0.0,1.0,0.0,0,0.27740,0.0,NaN,NaN,NaN,3.000000,0.277400
1,2022-07-25,ai,2,0.6115,0.000000,1.0,0.0,0.0,0,0.18340,0.0,3.0,NaN,-0.333333,2.500000,0.230400
2,2022-09-19,ai,6,-0.6044,0.035602,0.0,1.0,0.0,0,0.18135,0.0,2.0,NaN,2.000000,3.666667,0.214050
3,2022-09-26,ai,3,-0.7845,0.000000,0.0,1.0,0.0,0,0.23530,0.0,6.0,NaN,-0.500000,3.500000,0.219362
4,2022-10-03,ai,3,-0.7845,0.000000,0.0,1.0,0.0,0,0.23530,0.0,3.0,3.0,0.000000,3.500000,0.208838
5,2022-10-17,ai,3,-0.9153,0.000000,0.0,1.0,0.0,0,0.27460,0.0,3.0,2.0,0.000000,3.750000,0.231637
6,2022-10-24,ai,3,0.4588,0.000000,1.0,0.0,0.0,0,0.13760,0.0,3.0,6.0,0.000000,3.000000,0.220700
7,2022-10-31,ai,3,0.8689,0.000000,1.0,0.0,0.0,0,0.26070,0.0,3.0,3.0,0.000000,3.000000,0.227050
8,2022-11-07,ai,1,0.1027,0.000000,1.0,0.0,0.0,0,0.03080,0.0,3.0,3.0,-0.666667,2.500000,0.175925
9,2022-11-14,ai,2,0.8385,0.000000,1.0,0.0,0.0,0,0.25160,0.0,1.0,3.0,1.000000,2.250000,0.170175


In [5]:
# Pivot to wide format for modeling
# Keep volume and average risk index in separate matrices.
ts_wide_count = ts.pivot_table(index='week', columns='primary_topic', values='article_count', fill_value=0)
ts_wide_risk = ts.pivot_table(index='week', columns='primary_topic', values='risk_index_mean', fill_value=0)

# Prefix columns so both can live in one wide table.
ts_wide_count = ts_wide_count.add_prefix('count__')
ts_wide_risk = ts_wide_risk.add_prefix('risk__')

ts_wide = ts_wide_count.join(ts_wide_risk, how='outer').sort_index()

print('Wide format shape:', ts_wide.shape)
ts_wide.head()

Wide format shape: (219, 34)


primary_topic,count__ai,count__climate_weather,count__commodities_metals,count__cybersecurity,count__defense_security,count__elections_governance,count__energy_markets,count__finance_banking,count__food_water_security,count__geopolitics_conflict,...,risk__finance_banking,risk__food_water_security,risk__geopolitics_conflict,risk__labor_social,risk__macroeconomy,risk__natural_disasters,risk__public_health,risk__supply_chain,risk__technology_policy,risk__trade_industry
week,,,,,,,,,,,,,,,,,,,,,
2021-10-25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.0232,0.0,0.0000,0.0,0.0,0.0,0.0,0.0
2021-11-22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.2556,0.0,0.0000,0.0,0.0,0.0,0.0,0.0
2022-01-24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0000,0.0,0.1532,0.0,0.0,0.0,0.0,0.0
2022-01-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.1672,0.0,0.0000,0.0,0.0,0.0,0.0,0.0
2022-02-07,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0


In [7]:
ts.to_csv('data/timeseries.csv', index=False)
ts_wide.to_csv('data/timeseries_wide.csv')

print('Saved: data/timeseries.csv and data/timeseries_wide.csv')
print('\nTimeseries columns:')
print(ts.columns.tolist())

Saved: data/timeseries.csv and data/timeseries_wide.csv

Timeseries columns:
['week', 'primary_topic', 'article_count', 'sentiment_mean', 'sentiment_std', 'positive_ratio', 'negative_ratio', 'risk_score_mean', 'risk_score_max', 'risk_index_mean', 'high_risk_share', 'article_count_lag1', 'article_count_lag4', 'article_count_pct_change_1w', 'article_count_roll4_mean', 'risk_index_roll4_mean']
